<a href="https://colab.research.google.com/github/ingkapat/Thai-Scam-Call-Detector/blob/main/audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Install
!pip install -q google-cloud-texttospeech pydub
!apt-get install -y ffmpeg -qq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.1/200.1 kB 3.8 MB/s eta 0:00:00


In [ ]:
# Cell 2: Upload Service Account JSON key
# วิธีได้: console.cloud.google.com > IAM > Service Accounts > Create Key > JSON
from google.colab import files
uploaded = files.upload()
KEY_FILENAME = list(uploaded.keys())[0]
import os
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = f'/content/{KEY_FILENAME}'
print(f'Key: {KEY_FILENAME}')

Saving gen-lang-client-0115082914-75394e6ab8b5.json to gen-lang-client-0115082914-75394e6ab8b5.json
Key: gen-lang-client-0115082914-75394e6ab8b5.json


In [ ]:
# Cell 3: Upload dataset_cleaned.jsonl
uploaded = files.upload()
INPUT_FILENAME = list(uploaded.keys())[0]
print(f'Dataset: {INPUT_FILENAME}')

Saving dataset_cleaned.jsonl to dataset_cleaned.jsonl
Dataset: dataset_cleaned.jsonl


In [ ]:
# Cell 4: Config (Chirp3-HD quota 3000/min แล้ว)
import json, random, io
from pydub import AudioSegment
from google.cloud import texttospeech
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

OUT_DIR_FULL  = '/content/wav_output_full'    # เสียงเต็ม (raw)
OUT_DIR_15S   = '/content/wav_output_15s'     # เสียง trim/pad 15s

START_IDX     = 0
END_IDX       = 22156

FIX_DURATION  = 15000   # 15s
SILENCE_MS    = 500
MAX_WORKERS   = 30

os.makedirs(OUT_DIR_FULL, exist_ok=True)
os.makedirs(OUT_DIR_15S,  exist_ok=True)

_local = threading.local()
def get_client():
    if not hasattr(_local, 'client'):
        _local.client = texttospeech.TextToSpeechClient()
    return _local.client

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


In [ ]:
# Cell 5: Voice pool (Chirp3-HD ทั้งหมด 30 เสียง)
VOICE_POOL = [
    {'name': 'th-TH-Chirp3-HD-Kore',         'gender': 'F'},
    {'name': 'th-TH-Chirp3-HD-Aoede',        'gender': 'F'},
    {'name': 'th-TH-Chirp3-HD-Leda',         'gender': 'F'},
    {'name': 'th-TH-Chirp3-HD-Callirrhoe',   'gender': 'F'},
    {'name': 'th-TH-Chirp3-HD-Autonoe',      'gender': 'F'},
    {'name': 'th-TH-Chirp3-HD-Despina',      'gender': 'F'},
    {'name': 'th-TH-Chirp3-HD-Erinome',      'gender': 'F'},
    {'name': 'th-TH-Chirp3-HD-Laomedeia',    'gender': 'F'},
    {'name': 'th-TH-Chirp3-HD-Pulcherrima',  'gender': 'F'},
    {'name': 'th-TH-Chirp3-HD-Sulafat',      'gender': 'F'},
    {'name': 'th-TH-Chirp3-HD-Achernar',     'gender': 'F'},
    {'name': 'th-TH-Chirp3-HD-Gacrux',       'gender': 'F'},
    {'name': 'th-TH-Chirp3-HD-Vindemiatrix', 'gender': 'F'},
    {'name': 'th-TH-Chirp3-HD-Zephyr',       'gender': 'F'},
    {'name': 'th-TH-Chirp3-HD-Puck',         'gender': 'M'},
    {'name': 'th-TH-Chirp3-HD-Charon',       'gender': 'M'},
    {'name': 'th-TH-Chirp3-HD-Fenrir',       'gender': 'M'},
    {'name': 'th-TH-Chirp3-HD-Orus',         'gender': 'M'},
    {'name': 'th-TH-Chirp3-HD-Enceladus',    'gender': 'M'},
    {'name': 'th-TH-Chirp3-HD-Iapetus',      'gender': 'M'},
    {'name': 'th-TH-Chirp3-HD-Umbriel',      'gender': 'M'},
    {'name': 'th-TH-Chirp3-HD-Algieba',      'gender': 'M'},
    {'name': 'th-TH-Chirp3-HD-Algenib',      'gender': 'M'},
    {'name': 'th-TH-Chirp3-HD-Achird',       'gender': 'M'},
    {'name': 'th-TH-Chirp3-HD-Alnilam',      'gender': 'M'},
    {'name': 'th-TH-Chirp3-HD-Rasalgethi',   'gender': 'M'},
    {'name': 'th-TH-Chirp3-HD-Sadachbia',    'gender': 'M'},
    {'name': 'th-TH-Chirp3-HD-Sadaltager',   'gender': 'M'},
    {'name': 'th-TH-Chirp3-HD-Schedar',      'gender': 'M'},
    {'name': 'th-TH-Chirp3-HD-Zubenelgenubi','gender': 'M'},
]
print(f'Voices: {len(VOICE_POOL)} (M={sum(1 for v in VOICE_POOL if v["gender"]=="M")}, F={sum(1 for v in VOICE_POOL if v["gender"]=="F")})')

Voices: 30 (M=16, F=14)


In [ ]:
# Cell 6: TTS function + helpers (save 2 versions: full + 15s trim)
import time

def tts_segment(text, voice_name, max_retries=5):
    for attempt in range(max_retries):
        try:
            client = get_client()
            response = client.synthesize_speech(
                input=texttospeech.SynthesisInput(text=text),
                voice=texttospeech.VoiceSelectionParams(language_code='th-TH', name=voice_name),
                audio_config=texttospeech.AudioConfig(
                    audio_encoding=texttospeech.AudioEncoding.LINEAR16,
                    sample_rate_hertz=24000,
                ),
            )
            return AudioSegment.from_wav(io.BytesIO(response.audio_content))
        except Exception as e:
            err_str = str(e)
            if '429' in err_str or 'RESOURCE_EXHAUSTED' in err_str or 'quota' in err_str.lower():
                time.sleep(5 * (2 ** attempt))
            else:
                raise
    raise RuntimeError(f'rate-limit exceeded')

def pick_voice_pair(rng):
    males   = [v for v in VOICE_POOL if v['gender'] == 'M']
    females = [v for v in VOICE_POOL if v['gender'] == 'F']
    caller = rng.choice(VOICE_POOL)
    user = rng.choice(females if caller['gender']=='M' else males)
    return caller, user

def fix_to_duration(audio, target_ms):
    if len(audio) > target_ms:
        return audio[:target_ms]
    return audio + AudioSegment.silent(duration=target_ms - len(audio))

_error_samples = []
_error_lock = threading.Lock()

def process_conversation(idx, item, silence):
    fname = f'conv_{idx:05d}_label{item["label"]}.wav'
    fpath_full = os.path.join(OUT_DIR_FULL, fname)
    fpath_15s  = os.path.join(OUT_DIR_15S,  fname)

    # skip ถ้าทั้ง 2 ไฟล์มีอยู่แล้ว
    if os.path.exists(fpath_full) and os.path.exists(fpath_15s):
        return ('skip', fname, None)

    rng = random.Random(idx)
    caller_v, user_v = pick_voice_pair(rng)
    voice_map = {'caller': caller_v, 'user': user_v}

    combined = AudioSegment.empty()
    for turn in item['turns']:
        v = voice_map.get(turn['speaker'], caller_v)
        try:
            seg = tts_segment(turn['text'], v['name'])
            combined += seg + silence
        except Exception as e:
            with _error_lock:
                if len(_error_samples) < 5:
                    _error_samples.append(f'{fname}: {str(e)[:120]}')
            return ('error', fname, str(e)[:80])

    if len(combined) == 0:
        return ('empty', fname, None)

    # Save 2 versions
    combined.export(fpath_full, format='wav')                 # เต็ม
    fix_to_duration(combined, FIX_DURATION).export(fpath_15s, format='wav')  # 15s

    return ('saved', fname, {
        'label': item['label'],
        'caller': caller_v['name'],
        'user': user_v['name'],
        'duration_ms': len(combined),
    })

In [ ]:
# Cell 7: Load dataset + ตัดช่วง
data = []
with open(INPUT_FILENAME, 'r', encoding='utf-8') as f:
    for line in f:
        data.append(json.loads(line.strip()))

samples = data[START_IDX:END_IDX]
print(f'Total dataset: {len(data)}')
print(f'Range        : {START_IDX} → {END_IDX}')
print(f'Will process : {len(samples)} samples')

Total dataset: 21352
Range        : 0 → 22156
Will process : 21352 samples


In [ ]:
# Cell 8: Generate parallel
import time
silence = AudioSegment.silent(duration=SILENCE_MS)

_error_samples.clear()
saved_count = 0
skip_count  = 0
error_count = 0
metadata    = []
start = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(process_conversation, START_IDX+idx+1, item, silence): idx
               for idx, item in enumerate(samples)}

    for i, future in enumerate(as_completed(futures), 1):
        status, fname, info = future.result()
        if status == 'saved':
            saved_count += 1
            metadata.append({'file': fname, **info})
        elif status == 'skip':
            skip_count += 1
        else:
            error_count += 1

        if i % 50 == 0 or i == len(samples):
            elapsed = time.time() - start
            rate = i / elapsed * 60
            eta = (len(samples) - i) / max(rate/60, 0.1) / 60
            print(f'[{i}/{len(samples)}] saved={saved_count} skip={skip_count} err={error_count} | {rate:.0f}/min | ETA {eta:.1f}min')

if _error_samples:
    print(f'\n=== Error samples ===')
    for e in _error_samples:
        print(f'  {e}')

# metadata เก็บที่ root /content/ (ใช้ร่วมกัน)
meta_path = '/content/metadata.json'
existing_meta = []
if os.path.exists(meta_path):
    with open(meta_path, 'r', encoding='utf-8') as f:
        existing_meta = json.load(f)
all_meta = existing_meta + metadata
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(all_meta, f, ensure_ascii=False, indent=2)

elapsed = time.time() - start
print(f'\nDone in {elapsed/60:.1f} min | saved={saved_count} skip={skip_count} err={error_count}')
print(f'Full audio  → {OUT_DIR_FULL}')
print(f'15s audio   → {OUT_DIR_15S}')
print(f'Metadata    → {meta_path}')

[50/21352] saved=46 skip=0 err=4 | 400/min | ETA 53.2min
[100/21352] saved=95 skip=0 err=5 | 486/min | ETA 43.7min
[150/21352] saved=145 skip=0 err=5 | 534/min | ETA 39.7min
[200/21352] saved=194 skip=0 err=6 | 557/min | ETA 37.9min
[250/21352] saved=243 skip=0 err=7 | 579/min | ETA 36.5min
[300/21352] saved=293 skip=0 err=7 | 269/min | ETA 78.3min
[350/21352] saved=343 skip=0 err=7 | 215/min | ETA 97.7min
[400/21352] saved=393 skip=0 err=7 | 223/min | ETA 94.1min
[450/21352] saved=443 skip=0 err=7 | 239/min | ETA 87.4min
[500/21352] saved=493 skip=0 err=7 | 256/min | ETA 81.5min
[550/21352] saved=542 skip=0 err=8 | 270/min | ETA 77.0min
[600/21352] saved=592 skip=0 err=8 | 284/min | ETA 73.1min
[650/21352] saved=642 skip=0 err=8 | 296/min | ETA 69.9min
[700/21352] saved=691 skip=0 err=9 | 309/min | ETA 66.8min
[750/21352] saved=741 skip=0 err=9 | 323/min | ETA 63.8min
[800/21352] saved=791 skip=0 err=9 | 333/min | ETA 61.7min
[850/21352] saved=841 skip=0 err=9 | 344/min | ETA 59.7min


In [1]:
import os
print('wav_output_full:', len(os.listdir('/content/wav_output_full')) if os.path.exists('/content/wav_output_full') else 'หายแล้ว')
print('wav_output_15s :', len(os.listdir('/content/wav_output_15s')) if os.path.exists('/content/wav_output_15s') else 'หายแล้ว')
print('metadata.json  :', 'มี' if os.path.exists('/content/metadata.json') else 'หาย')

wav_output_full: 21287
wav_output_15s : 21287
metadata.json  : มี


In [5]:
# Cell 9 เร็วขึ้น (pydub):
from pydub import AudioSegment
from concurrent.futures import ThreadPoolExecutor
from glob import glob
import os, shutil

OUT_DIR_15S  = '/content/wav_output_15s'
OUT_DIR_FULL = '/content/wav_output_full'
MP3_DIR_15S  = '/content/mp3_15s'
MP3_DIR_FULL = '/content/mp3_full'
os.makedirs(MP3_DIR_15S,  exist_ok=True)
os.makedirs(MP3_DIR_FULL, exist_ok=True)

def convert(args):
    wav, out_dir = args
    mp3 = os.path.join(out_dir, os.path.basename(wav).replace('.wav', '.mp3'))
    if os.path.exists(mp3):
        return
    AudioSegment.from_wav(wav).export(mp3, format='mp3', bitrate='64k')

for src, dst in [(OUT_DIR_15S, MP3_DIR_15S), (OUT_DIR_FULL, MP3_DIR_FULL)]:
    wavs = glob(f'{src}/*.wav')
    print(f'{src}: {len(wavs)} files')
    with ThreadPoolExecutor(max_workers=16) as ex:
        list(ex.map(convert, [(w, dst) for w in wavs]))

shutil.make_archive('/content/mp3_15s',  'zip', MP3_DIR_15S)
shutil.make_archive('/content/mp3_full', 'zip', MP3_DIR_FULL)

print(f'mp3_15s.zip : {os.path.getsize("/content/mp3_15s.zip")/1024/1024:.1f} MB')
print(f'mp3_full.zip: {os.path.getsize("/content/mp3_full.zip")/1024/1024:.1f} MB')

from google.colab import files
files.download('/content/mp3_15s.zip')
files.download('/content/mp3_full.zip')

/content/wav_output_15s: 21287 files
/content/wav_output_full: 21287 files
mp3_15s.zip : 1892.1 MB
mp3_full.zip: 2467.7 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>